# LLaMA 3.1 on SageMaker

This notebook shows how to:

- Deploy a Hugging Face LLaMA 3.1 model on Amazon SageMaker  
- Run a quick test inference  
- **Always tear down the endpoint** to avoid idle GPU charges


## 1) Config
- Default model: Qwen2.5-3B Instruct (good quality/cost)
- Instance: ml.g5.xlarge (A10G 24GB)
- Token limits: enforce MAX_INPUT_TOKENS < MAX_TOTAL_TOKENS (TGI requirement)

In [3]:
REGION = "us-west-2"

import boto3, botocore, os, json, time, typing, uuid
from sagemaker import get_execution_role
from sagemaker.huggingface import HuggingFaceModel
from sagemaker.async_inference import AsyncInferenceConfig
from sagemaker.session import Session

# Session/clients
boto_sess = boto3.Session(region_name=REGION)
sm  = boto_sess.client("sagemaker")
rt  = boto_sess.client("sagemaker-runtime")
s3  = boto_sess.client("s3")
logs = boto_sess.client("logs")
appscaling = boto_sess.client("application-autoscaling")
cw  = boto_sess.client("cloudwatch")
sts = boto_sess.client("sts")
sagemaker_sess = Session(boto_session=boto_sess)

ACCOUNT = sts.get_caller_identity()["Account"]
ROLE = get_execution_role()

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
ENDPOINT_NAME = "neuro-rag-async"
VARIANT_NAME  = "AllTraffic"
INSTANCE_TYPE = "ml.g5.xlarge"


MAX_INPUT_TOKENS = 1536
MAX_TOTAL_TOKENS = 2048

# Async I/O (S3) — standard SageMaker bucket in this region
BUCKET = sagemaker_sess.default_bucket()

INPUT_PREFIX  = f"async-inputs/{ENDPOINT_NAME}"
OUTPUT_PREFIX = f"async-outputs/{ENDPOINT_NAME}"

# TGI DLC image
IMAGE_URI = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/huggingface-pytorch-tgi-inference:2.4.0-tgi2.4.0-gpu-py311-cu124-ubuntu22.04"

print("Region:", REGION)
print("Bucket:", BUCKET)
print("Model:", MODEL_ID)
print("Image:", IMAGE_URI)
print("Endpoint:", ENDPOINT_NAME)

Region: us-west-2
Account: 575935529773
Role: arn:aws:iam::575935529773:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole
Bucket: sagemaker-us-west-2-575935529773
Model: Qwen/Qwen2.5-3B-Instruct
Image: 763104351884.dkr.ecr.us-west-2.amazonaws.com/huggingface-pytorch-tgi-inference:2.4.0-tgi2.4.0-gpu-py311-cu124-ubuntu22.04
Endpoint: neuro-rag-async


## 2) Helpers
- `kill`: idempotent teardown (endpoint → config → model)
- `wait_status`: waits until InService/Failed
- `wait_deleted`: waits until endpoint fully deleted
- `tail_logs`: quick peek at CloudWatch logs

In [5]:
def safe_call(fn: typing.Callable, **kw):
    try:
        return fn(**kw)
    except botocore.exceptions.ClientError as e:
        code = e.response.get("Error", {}).get("Code")
        if code in {"ValidationException", "ResourceNotFound"}:
            return None
        raise

def kill(name: str):
    print(f"Deleting endpoint (if exists): {name}")
    safe_call(sm.delete_endpoint, EndpointName=name)
    print(f"Deleting endpoint config (if exists): {name}")
    safe_call(sm.delete_endpoint_config, EndpointConfigName=name)
    print(f"Deleting model (if exists): {name}")
    safe_call(sm.delete_model, ModelName=name)

def wait_status(name: str, desired=("InService",), fail=("Failed",), timeout_min=45):
    last = None
    t0 = time.time()
    while True:
        d = sm.describe_endpoint(EndpointName=name)
        st = d["EndpointStatus"]
        if st != last:
            print("Endpoint status:", st)
            last = st
            if st in fail:
                print("FailureReason:\n", d.get("FailureReason"))
        if st in desired or st in fail:
            return st
        if time.time() - t0 > timeout_min*60:
            print("Timeout waiting for", desired)
            return "TimedOut"
        time.sleep(10)

def wait_deleted(name: str, timeout_min=15):
    t0 = time.time()
    while True:
        try:
            sm.describe_endpoint(EndpointName=name)
        except botocore.exceptions.ClientError as e:
            if e.response.get("Error", {}).get("Code") == "ValidationException":
                print("Endpoint deleted:", name)
                return True
            raise
        if time.time() - t0 > timeout_min*60:
            print("Timed out waiting for deletion.")
            return False
        time.sleep(8)

def tail_logs(endpoint_name: str, seconds=1800, lines=120):
    group = f"/aws/sagemaker/Endpoints/{endpoint_name}"
    start = int((time.time() - seconds) * 1000)
    try:
        streams = logs.describe_log_streams(
            logGroupName=group, orderBy="LastEventTime", descending=True
        ).get("logStreams", [])
        if not streams:
            print("No log streams yet.")
            return
        for s in streams[:2]:
            evs = logs.get_log_events(
                logGroupName=group, logStreamName=s["logStreamName"], startTime=start
            )["events"]
            print(f"\n--- {s['logStreamName']} ---")
            for m in evs[-lines:]:
                print(m["message"].rstrip())
    except logs.exceptions.ResourceNotFoundException:
        print("No CloudWatch log group found (endpoint may not have started yet).")


## 3) Build TGI environment

In [6]:
env = {
    "HF_MODEL_ID": MODEL_ID,
    "HF_TASK": "text-generation",
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "MAX_INPUT_TOKENS": str(MAX_INPUT_TOKENS),
    "MAX_TOTAL_TOKENS": str(MAX_TOTAL_TOKENS),
}

print({k: env[k] for k in ["HF_MODEL_ID", "MAX_INPUT_TOKENS", "MAX_TOTAL_TOKENS"]})

{'HF_MODEL_ID': 'Qwen/Qwen2.5-3B-Instruct', 'MAX_INPUT_TOKENS': '1536', 'MAX_TOTAL_TOKENS': '2048'}


## 4) Clean slate & DEPLOY as **Async**
- Creates the model + endpoint config + async endpoint
- Waits for **InService**

In [7]:
kill(ENDPOINT_NAME)

hf_model = HuggingFaceModel(
    image_uri=IMAGE_URI,
    role=ROLE,
    env=env,
    sagemaker_session=sagemaker_sess,
)

async_cfg = AsyncInferenceConfig(
    output_path=f"s3://{BUCKET}/{OUTPUT_PREFIX}",
)

predictor = hf_model.deploy(
    endpoint_name=ENDPOINT_NAME,
    initial_instance_count=1,  # starts warm; autoscaling will handle scale-down to 0 when idle
    instance_type=INSTANCE_TYPE,
    async_inference_config=async_cfg,
    container_startup_health_check_timeout=1800,
    wait=True,
)
print("Async endpoint created. Waiting for InService...")
print("Endpoint:", ENDPOINT_NAME)

st = wait_status(ENDPOINT_NAME, desired=("InService",), fail=("Failed",), timeout_min=30)
if st != "InService":
    print("Logs for diagnosis:")
    tail_logs(ENDPOINT_NAME, seconds=3600, lines=200)
    raise RuntimeError(f"Endpoint not ready (status={st}).")


Deleting endpoint (if exists): neuro-rag-async
Deleting endpoint config (if exists): neuro-rag-async
Deleting model (if exists): neuro-rag-async
-----------!✅ Async endpoint created. Waiting for InService...
Endpoint: neuro-rag-async
Endpoint status: InService


## 5) AUTOSCALING: enable **scale-to-zero**
- Target-tracking: matches capacity to backlog
- Step-scaling + alarm: wakes from 0→1 when queued work appears


In [8]:
resource_id = f"endpoint/{ENDPOINT_NAME}/variant/{VARIANT_NAME}"

# Allow MinCapacity=0 (scale-to-zero)
appscaling.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=0,
    MaxCapacity=2,
)

# Target-tracking policy
policy_tt = appscaling.put_scaling_policy(
    PolicyName="AsyncBacklogTargetTracking",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 5.0,  # ~5 queued per instance
        "CustomizedMetricSpecification": {
            "MetricName": "ApproximateBacklogSizePerInstance",
            "Namespace": "AWS/SageMaker",
            "Dimensions": [{"Name": "EndpointName", "Value": ENDPOINT_NAME}],
            "Statistic": "Average",
        }
    },
)
print("Target-tracking policy:", policy_tt["PolicyARN"])

# Step-scaling + alarm to wake immediately from 0
policy_step = appscaling.put_scaling_policy(
    PolicyName="HasBacklogWithoutCapacity-ScalingPolicy",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="StepScaling",
    StepScalingPolicyConfiguration={
        "AdjustmentType": "ChangeInCapacity",
        "MetricAggregationType": "Average",
        "Cooldown": 300,
        "StepAdjustments": [{"MetricIntervalLowerBound": 0, "ScalingAdjustment": 1}],
    },
)
cw.put_metric_alarm(
    AlarmName=f"{ENDPOINT_NAME}-HasBacklogWithoutCapacity",
    MetricName="HasBacklogWithoutCapacity",
    Namespace="AWS/SageMaker",
    Statistic="Average",
    EvaluationPeriods=2,
    DatapointsToAlarm=2,
    Threshold=1,
    ComparisonOperator="GreaterThanOrEqualToThreshold",
    TreatMissingData="missing",
    Dimensions=[{"Name": "EndpointName", "Value": ENDPOINT_NAME}],
    Period=60,
    AlarmActions=[policy_step["PolicyARN"]],
)
print("Autoscaling to zero configured (Min=0; backlog wakes it up).")


Target-tracking policy: arn:aws:autoscaling:us-west-2:575935529773:scalingPolicy:7c90d564-1e76-4281-bee8-be8b54bd6984:resource/sagemaker/endpoint/neuro-rag-async/variant/AllTraffic:policyName/AsyncBacklogTargetTracking
✅ Autoscaling to zero configured (Min=0; backlog wakes it up).


## 6) Async client helper (S3 upload + poll for result)
Use this to send prompts and wait for the generated text.


In [9]:
def invoke_async_tgi(prompt: str,
                     max_new_tokens: int = 200,
                     temperature: float = 0.7,
                     timeout_s: int = 180) -> str:
    """Uploads request to S3, calls invoke_endpoint_async, polls S3 for result, returns generated_text."""
    req = {"inputs": prompt, "parameters": {"max_new_tokens": max_new_tokens, "temperature": temperature}}
    rid = str(uuid.uuid4())
    in_key = f"{INPUT_PREFIX}/{rid}.json"

    # Mark the S3 object as JSON
    s3.put_object(
        Bucket=BUCKET,
        Key=in_key,
        Body=json.dumps(req).encode("utf-8"),
        ContentType="application/json",
    )

    resp = rt.invoke_endpoint_async(
        EndpointName=ENDPOINT_NAME,
        InputLocation=f"s3://{BUCKET}/{in_key}",
        ContentType="application/json",
    )

    out_uri = resp["OutputLocation"]
    _, _, rest = out_uri.partition("s3://")
    out_bucket, _, out_key = rest.partition("/")

    t0 = time.time()
    backoff = 1.0
    while True:
        try:
            obj = s3.get_object(Bucket=out_bucket, Key=out_key)
            data = json.loads(obj["Body"].read())
            if isinstance(data, list) and data and "generated_text" in data[0]:
                return data[0]["generated_text"]
            return data.get("generated_text", json.dumps(data))
        except s3.exceptions.NoSuchKey:
            pass
        except botocore.exceptions.ClientError as e:
            if e.response.get("Error", {}).get("Code") != "NoSuchKey":
                raise
        if time.time() - t0 > timeout_s:
            raise TimeoutError(f"Async result not ready after {timeout_s}s: {out_uri}")
        time.sleep(backoff)
        backoff = min(backoff * 1.5, 4.0)


## 7) Quick test generation
(Keep `max_new_tokens` modest to control cost/latency)


In [10]:
test_prompt = (
    "Write a ~40 word opening scenario set in 2075 about memory implants and ethics. "
    "Output JSON with keys: scenario_text, choices (3 items), citations (empty array is fine)."
)
txt = invoke_async_tgi(test_prompt, max_new_tokens=180, temperature=0.7, timeout_s=40)
print("Model output:\n", txt)


Model output:
 Write a ~40 word opening scenario set in 2075 about memory implants and ethics. Output JSON with keys: scenario_text, choices (3 items), citations (empty array is fine). JSON:
{
    "scenario_text": "In 2075, society heavily relies on advanced memory implants to enhance learning and recall, but ethical debates rage over the extent of personal freedom versus improved cognitive abilities.",
    "choices": [
        "Some citizens argue that memory implants are enslaving the human mind.",
        "Others believe that they are necessary for a functioning and competitive society.",
        "Experts suggest a balanced approach, integrating implants selectively and transparently."
    ],
    "citations": []
} This JSON structure encapsulates a scenario set in 2075 regarding memory implants with their ethical implications, along with potential viewpoints on the matter. The "choices" array provides three perspectives to consider, reflecting the complexity of the issue highlighted

## 8) Teardown
This deletes the endpoint, config, and model. Safe to re-run.


In [26]:
print("Tearing down:", ENDPOINT_NAME)
kill(ENDPOINT_NAME)
wait_deleted(ENDPOINT_NAME)

safe_call(sm.delete_endpoint_config, EndpointConfigName=ENDPOINT_NAME)
safe_call(sm.delete_model, ModelName=ENDPOINT_NAME)
print("Cleanup done.")


Tearing down: neuro-rag-async
Deleting endpoint (if exists): neuro-rag-async
Deleting endpoint config (if exists): neuro-rag-async
Deleting model (if exists): neuro-rag-async
✅ Endpoint deleted: neuro-rag-async
Cleanup done.
